In [1]:
import uuid

import httpx
from a2a.client import A2ACardResolver, create_client
from a2a.types import (
    AgentCard,
    Message,
    Part,
    Role,
    SendMessageRequest
)
from google.protobuf.json_format import MessageToJson

### Test the connection

In [2]:
BASE_URL = "http://localhost:10001"
PUBLIC_AGENT_CARD_PATH = "/.well-known/agent-card.json"

In [3]:
async with httpx.AsyncClient() as httpx_client:

    resolver = A2ACardResolver(
        httpx_client=httpx_client,
        base_url=BASE_URL
    )

    try:
        print(f"Fetching public agent card from: {BASE_URL}{PUBLIC_AGENT_CARD_PATH}")
        public_card = await resolver.get_agent_card()
        print("Fetched public agent card")
        print(MessageToJson(public_card, indent=2))

    except Exception as e:
        print(f"Error fetching public agent card: {e}")

Fetching public agent card from: http://localhost:10001/.well-known/agent-card.json
Fetched public agent card
{
  "name": "warehouse_manager_agent",
  "description": "The user is asking to reserve items from the warehouses or about availability of the items in warehouses.",
  "supportedInterfaces": [
    {
      "url": "http://localhost:10001/",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "skills": [
    {
      "id": "ABC",
      "name": "Check Availability",
      "description": "Check availability of items across warehouses.",
      "tags": [
        "availability",
        "warehouse"
      ],
      "examples": [
        "What is the availability of the item 123?"
      ]
    },
    {
      "id": "DEF",
      "name": "Reserve Items",
      "description": "Reserve items from multiple warehouses in 

### Run commands against the Agent

In [5]:
timeout_config = httpx.Timeout(
    connect=10.0,   # Connection timeout
    read=100.0,     # Read timeout (important for long-running operations)
    write=10.0,     # Write timeout
    pool=10.0       # Pool timeout
)

client = await create_client(
    agent=BASE_URL,
    resolver_http_kwargs={"timeout": timeout_config}
)

print("A2AClient initialized")

message_id = str(uuid.uuid4())
message_payload = Message(
    role=Role.ROLE_USER,
    message_id=message_id,
    parts=[Part(text="Hello, how are you?")],
)
request = SendMessageRequest(message=message_payload)

async for response in client.send_message(request):
    print(response)

A2AClient initialized
task {
  id: "ec6a7156-f76a-4337-b8cb-c0571c968dc9"
  context_id: "b4a09738-9961-40ea-8da1-2a8b0c9f21d1"
  status {
    state: TASK_STATE_SUBMITTED
  }
}

status_update {
  task_id: "ec6a7156-f76a-4337-b8cb-c0571c968dc9"
  context_id: "b4a09738-9961-40ea-8da1-2a8b0c9f21d1"
  status {
    state: TASK_STATE_WORKING
    timestamp {
      seconds: 1786217068
      nanos: 86993000
    }
  }
}

artifact_update {
  task_id: "ec6a7156-f76a-4337-b8cb-c0571c968dc9"
  context_id: "b4a09738-9961-40ea-8da1-2a8b0c9f21d1"
  artifact {
    artifact_id: "a7e5ef57-1db7-4b0e-8bd2-f2bf82bf53df"
    parts {
      text: "Actions performed: none.\n\nI’m doing well, thank you. How can I help with warehouse availability or reservations?"
    }
  }
}

status_update {
  task_id: "ec6a7156-f76a-4337-b8cb-c0571c968dc9"
  context_id: "b4a09738-9961-40ea-8da1-2a8b0c9f21d1"
  status {
    state: TASK_STATE_COMPLETED
    timestamp {
      seconds: 1786217069
      nanos: 905232000
    }
  }
}



In [6]:
async def run_a2a_warehouse_agent(query: str):

    timeout_config = httpx.Timeout(
        connect=10.0,   # Connection timeout
        read=100.0,     # Read timeout (important for long-running operations)
        write=10.0,     # Write timeout
        pool=10.0       # Pool timeout
    )

    client = await create_client(
        agent=BASE_URL,
        resolver_http_kwargs={"timeout": timeout_config}
    )


    message_id = str(uuid.uuid4())
    message_payload = Message(
        role=Role.ROLE_USER,
        message_id=message_id,
        parts=[Part(text=query)],
    )
    request = SendMessageRequest(message=message_payload)

    final_text_parts = []

    async for response in client.send_message(request):
        if response.HasField("artifact_update"):
            for part in response.artifact_update.artifact.parts:
                if part.text:
                    final_text_parts.append(part.text)

    final_response = "".join(final_text_parts)

    return final_response

In [7]:
answer_1 = await run_a2a_warehouse_agent("What is the availability of B0B63FT91X in all of your warehouses?")

In [8]:
print(answer_1)

Actions performed:
1. Checked availability for product **B0B63FT91X** across all warehouses.

Availability result:
- **Berlin Distribution Center (DE-BER-01)** — 88 available
- **Lyon Regional Warehouse (FR-LYO-01)** — 47 available
- **Munich Logistics Hub (DE-MUN-01)** — 59 available
- **Paris Central Depot (FR-PAR-01)** — 0 available
- **Marseille Mediterranean Hub (FR-MAR-01)** — 38 available
- **Hamburg North Warehouse (DE-HAM-01)** — 53 available

Summary:
- The item is **available in multiple warehouses**
- It can be **fully fulfilled** from any of these warehouses
- No reservation was made


In [9]:
answer_1 = await run_a2a_warehouse_agent("can you reserve 10 of B0B63FT91X in Hamburg?")

In [10]:
print(answer_1)

Actions performed:
- Checked warehouse availability for 10× B0B63FT91X.
- Confirmed the order can be fully fulfilled.
- Reserved 10× B0B63FT91X from Hamburg North Warehouse (DE-HAM-01).

Reservation status: successful.
